# Hybrid Stacking Framework for Financial Fraud Detection
## Production-Ready Stacked Generalization Pipeline (Level 0 + Level 1)

This notebook implements a **leakage-free stacked generalization** pipeline for credit-card
fraud detection across three benchmark datasets:

| Dataset                     | Source                                                                 |
|-----------------------------|------------------------------------------------------------------------|
| **Sparkov**                 | Synthetic transactions generated via the Sparkov profiling tool         |
| **IEEE-CIS**                | Vesta Corporation real-world e-commerce fraud (Kaggle)                 |
| **European Credit Card 2013**| ULB ML benchmark — anonymized European cardholders (Kaggle)           |

### Architecture
1. **Level 0 — Base Layer:** a diverse pool of 8+ estimators spanning four families:
   - **Deep Learning sequence models:** CNN, LSTM, Vanilla RNN, GRU (TF/Keras)
   - **Ultra-Fast estimators:** LightGBM, GaussianNB/ComplementNB, SGDClassifier
   - **Tree Boosting:** XGBoost, CatBoost
   - **Linear / Distance:** LogisticRegression (L1/ElasticNet), LinearSVC, KNN
2. **Level 1 — Meta-Learner:** a regularized `LogisticRegression(penalty='l2')`
   (or `RidgeClassifier` / shallow `RandomForestClassifier` as alternatives) that learns
   optimal blending weights from out-of-fold (OOF) base predictions.

### Anti-Leakage Guarantees
- **Stratified 5-Fold CV** generates OOF features strictly on the training partition.
- **`imblearn.pipeline.Pipeline`** wraps every preprocessor so statistics are fit on
  the *internal training fold only* — no optimism bias.
- **Validation set (`_val.csv`) is treated as strictly isolated** — never used for any
  decision; only sanity-check predictions are generated.
- **Test set (`_test.csv`) is single-use** — evaluated once, at the very end, by the
  fully-fitted Level-1 stacker.
- **Determinism** is enforced via global seeding (`random`, `numpy`, `tensorflow`).

In [1]:
import sys, os
# Redirect stderr to /dev/null – silences all C++ noise from TensorFlow
sys.stderr = open(os.devnull, 'w')
# =====================================================================
#  Imports
# =====================================================================
import gc
import json
import random
import warnings
import contextlib
import shutil
import time
import joblib           
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional, Callable

import numpy as np
import pandas as pd

# ---- scikit-learn core -------------------------------------------------
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import (
    LogisticRegression, SGDClassifier, RidgeClassifier
)
from sklearn.naive_bayes import GaussianNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest  
from sklearn.svm import OneClassSVM  
from sklearn.neighbors import LocalOutlierFactor
from sklearn.neural_network import MLPClassifier 
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix
)

# ---- imblearn (Pipeline that preserves SMOTE / undersampling) ---------
from imblearn.pipeline import Pipeline as ImbPipeline

# ---- Boosting libraries ------------------------------------------------
import lightgbm as lgb
from lightgbm import LGBMClassifier

import xgboost as xgb
from xgboost import XGBClassifier

# CatBoost is optional
try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

# ---- TensorFlow / Keras ------------------------------------------------
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, backend as K

# Silence TF / sklearn noise
warnings.filterwarnings('ignore')
# ---- TensorFlow runtime configuration --------------------------------
os.environ['TF_CPP_MIN_LOG_LEVEL']  = '3'
os.environ['CUDA_VISIBLE_DEVICES']  = '-1'
os.environ['TF_DETERMINISTIC_OPS']  = '1'

print(f"[deps] TensorFlow : {tf.__version__}")
print(f"[deps] CatBoost available: {HAS_CATBOOST}")

I0000 00:00:1786091481.598681 2724177 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786091481.599765 2724177 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786091481.831549 2724177 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1786091482.822258 2724177 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

[deps] TensorFlow : 2.21.0
[deps] CatBoost available: True


E0000 00:00:1786091483.933232 2724177 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/home/phd/Documents/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
E0000 00:00:1786092216.829243 2724177 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node Pa

In [2]:
import re

def safe_columns(df):
    """Replace special JSON‑unsafe characters in column names with underscores."""
    df = df.rename(columns=lambda c: re.sub(r'[^a-zA-Z0-9_]', '_', str(c)))
    return df

In [3]:
# =====================================================================
#  Global Seeding for Determinism
# =====================================================================

GLOBAL_SEED = 42

def set_global_seed(seed: int = GLOBAL_SEED) -> None:
    """Pin every RNG we touch to `seed`."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except Exception:
        pass

set_global_seed(GLOBAL_SEED)
print(f"[seed] Global random seed = {GLOBAL_SEED}")

[seed] Global random seed = 42


In [4]:
# =====================================================================
#  Configuration
# =====================================================================
CONFIG = {
    # --- Paths ----------------------------------------------------------
    "DATA_ROOT":   "./prepareddata",          # root holding dataset sub-folders
    "OUTPUT_DIR":  "./trained",
    "TARGET_COL":  "Fraud",                    # standardized target name

    # --- CV -------------------------------------------------------------
    "N_SPLITS":    5,
    "RANDOM_STATE": GLOBAL_SEED,

    # --- DL training ----------------------------------------------------
    "DL_EPOCHS":     100,        # cap; early-stopping kicks in earlier
    "DL_BATCH_SIZE": 256,
    "DL_VAL_SPLIT":  0.10,      # 10% sub-val inside each fold
    "DL_PATIENCE":   10,

    # --- OOF caching ----------------------------------------------------
    "CACHE_OOF":   True,        # save OOF matrices & test preds to disk
    "OVERWRITE":   False,       # skip variants that already have artifacts

    # --- Misc -----------------------------------------------------------
    "VERBOSE":     1,           # 0 silent, 1 progress, 2 per-fold
}

# Create output directory
Path(CONFIG["OUTPUT_DIR"]).mkdir(parents=True, exist_ok=True)
print(f"[config] DATA_ROOT={CONFIG['DATA_ROOT']}")
print(f"[config] OUTPUT_DIR={CONFIG['OUTPUT_DIR']}")
print(f"[config] TARGET_COL={CONFIG['TARGET_COL']}  N_SPLITS={CONFIG['N_SPLITS']}")

[config] DATA_ROOT=./prepareddata
[config] OUTPUT_DIR=./trained
[config] TARGET_COL=Fraud  N_SPLITS=5


In [5]:
# =====================================================================
#  Data Discovery
# =====================================================================


def parse_filename(stem: str) -> Optional[Dict[str, str]]:
    """Split '<sampler>--<featsel>--<dataset>--<timestamp>[_val|_test]'."""
    parts = stem.split("--")
    if len(parts) < 4:
        return None
    sampler, featsel, dataset = parts[0], parts[1], parts[2]
    tail = "--".join(parts[3:])           # in case timestamp itself contains '--'
    suffix = None
    for s in ("_val", "_test"):
        if tail.endswith(s):
            suffix = s
            tail = tail[: -len(s)]
            break
    return {"sampler": sampler, "featsel": featsel,
            "dataset": dataset, "timestamp": tail, "suffix": suffix}


def discover_datasets(data_root: str) -> Dict[str, Dict[str, Any]]:
    """
    Walk DATA_ROOT and return:
        {
          '<dataset>': {
              'train': [Path, ...],   # sorted list of training variants
              'val':   Path,
              'test':  Path,
          },
          ...
        }
    Scans recursively so files may live flat or under per-dataset sub-folders.
    """
    root = Path(data_root)
    if not root.exists():
        raise FileNotFoundError(
            f"DATA_ROOT '{data_root}' does not exist. "
            f"Run preparedata.ipynb first or set DATA_ROOT accordingly."
        )

    # ---- 1) bucket every CSV by its embedded dataset name --------------
    all_csvs = list(root.rglob("*.csv")) or list(root.glob("*.csv"))
    by_ds: Dict[str, List[Tuple[Path, Dict[str, str]]]] = {}
    for csv in all_csvs:
        info = parse_filename(csv.stem)
        if info is None:
            continue
        by_ds.setdefault(info["dataset"], []).append((csv, info))

    datasets: Dict[str, Dict[str, Any]] = {}
    for ds, files in by_ds.items():
        train_files = [p for p, i in files if i["suffix"] is None]
        val_file    = next((p for p, i in files if i["suffix"] == "_val"),  None)
        test_file   = next((p for p, i in files if i["suffix"] == "_test"), None)

        # ---- 2) graceful fallback for simpler val/test naming -----------
        if val_file is None:
            for pat in (f"{ds}_val.csv", f"{ds}--*_val.csv", f"{ds}-val.csv"):
                hits = list(root.rglob(pat))
                if hits:
                    val_file = hits[0]
                    break
        if test_file is None:
            for pat in (f"{ds}_test.csv", f"{ds}--*_test.csv", f"{ds}-test.csv"):
                hits = list(root.rglob(pat))
                if hits:
                    test_file = hits[0]
                    break

        if train_files and val_file and test_file:
            datasets[ds] = {"train": sorted(train_files),
                            "val":   val_file, "test": test_file}
            print(f"[discover] {ds:12s} -> {len(train_files)} train variants, "
                  f"val & test present")
        else:
            print(f"[discover][skip] {ds} "
                  f"(train={bool(train_files)}, val={bool(val_file)}, "
                  f"test={bool(test_file)})")

    if not datasets:
        raise RuntimeError(
            f"No usable datasets found under {data_root}. "
            f"Expected filenames like 'SMOTE--ANOVA_k5--Sparkov--20260628_184524.csv'"
        )
    return datasets


datasets = discover_datasets(CONFIG["DATA_ROOT"])
print(f"\n[discover] {len(datasets)} datasets ready: {list(datasets.keys())}")

[discover] EuropeanCard -> 180 train variants, val & test present
[discover] Sparkov      -> 180 train variants, val & test present
[discover] IEEE-CIS     -> 200 train variants, val & test present

[discover] 3 datasets ready: ['EuropeanCard', 'Sparkov', 'IEEE-CIS']


In [6]:
# =====================================================================
#  UnsupervisedAnomalyClassifier – sklearn‑compatible wrapper
# =====================================================================
# Wraps IsolationForest, OneClassSVM, LocalOutlierFactor so they provide
# predict_proba() (fraud = class 1).  The raw anomaly score is min‑max
# scaled on the training set and inverted to produce a pseudo‑probability.
# =====================================================================

class UnsupervisedAnomalyClassifier(BaseEstimator, ClassifierMixin):
    """
    Parameters
    ----------
    detector : object
        An unfitted sklearn anomaly detector (IsolationForest, OneClassSVM,
        LocalOutlierFactor with novelty=True).
    """
    def __init__(self, detector):
        self.detector = detector

    def fit(self, X, y=None):
        # Fit on all training data (labels ignored)
        self.detector_ = clone(self.detector)
        self.detector_.fit(X)
        # Compute raw scores on training data for later scaling
        train_scores = self._raw_score(X)
        self.score_min_ = np.min(train_scores)
        self.score_max_ = np.max(train_scores)
        self.classes_ = np.array([0, 1])          # 0 = normal, 1 = fraud
        return self

    def _raw_score(self, X):
        """Return the raw anomaly score (higher = more normal)."""
        if hasattr(self.detector_, "decision_function"):
            return self.detector_.decision_function(X)
        elif hasattr(self.detector_, "score_samples"):
            return self.detector_.score_samples(X)
        else:
            # fallback: inverse of predict (1 = inlier, -1 = outlier)
            return self.detector_.predict(X).astype(float)

    def predict_proba(self, X):
        raw = self._raw_score(X)
        eps = 1e-8
        # Scale to [0, 1] where 1 = most normal
        norm = np.clip((raw - self.score_min_) / (self.score_max_ - self.score_min_ + eps), 0, 1)
        fraud_prob = 1.0 - norm          # anomalous -> high fraud prob
        legit_prob = norm
        return np.vstack([legit_prob, fraud_prob]).T

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

In [7]:
# =====================================================================
#  apply_preprocessing – bring raw val/test into the training feature space
# =====================================================================
# This version handles unfitted imputers (e.g., when a dataset has zero
# categorical or zero numerical columns after dropping high‑missing values).
# =====================================================================

from sklearn.utils.validation import check_is_fitted

_ID_DROP_COLS = [
    "cc_num", "merchant", "nameOrig", "nameDest", "trans_num",
    "TransactionID", "card1", "addr1", "P_emaildomain", "R_emaildomain",
    "DeviceInfo"
]

# ---- helper: detect categorical / numerical columns (same logic as original) ----
def _detect_types(X: pd.DataFrame):
    """Return (cat_cols, num_cols) using the original heuristic."""
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    int_cols = X.select_dtypes(include=['int']).columns
    for c in int_cols:
        if c not in cat_cols and X[c].nunique() < 10:
            cat_cols.append(c)
            num_cols.remove(c)
    return cat_cols, num_cols


def _add_engineered_features(df: pd.DataFrame, dataset_name: str,
                             agg_stats: dict) -> pd.DataFrame:
    """Add time/log/aggregation features using pre‑computed tables."""
    df = df.copy()
    if dataset_name == "Sparkov":
        if "unix_time" in df.columns:
            trans_dt = pd.to_datetime(df["unix_time"], unit='s')
            df["hour"] = trans_dt.dt.hour
            df["dayofweek"] = trans_dt.dt.dayofweek
            df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
            df["month"] = trans_dt.dt.month
        if "amt" in df.columns:
            df["log_amt"] = np.log1p(df["amt"])
        if "cc_num" in df.columns and "sparkov_card_stats" in agg_stats:
            df = df.merge(agg_stats["sparkov_card_stats"], on="cc_num", how="left")
        if "merchant" in df.columns and "sparkov_merch_stats" in agg_stats:
            df = df.merge(agg_stats["sparkov_merch_stats"], on="merchant", how="left")
        if "merch_lat" in df.columns and "merch_long" in df.columns:
            def haversine_vectorised(lat1, lon1, lat2, lon2):
                lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
                dlat = lat2 - lat1
                dlon = lon2 - lon1
                a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
                c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
                return 6371.0 * c
            df["customer_merchant_dist"] = haversine_vectorised(
                df["lat"].values, df["long"].values,
                df["merch_lat"].values, df["merch_long"].values
            )
    elif dataset_name == "IEEE-CIS":
        if "TransactionDT" in df.columns:
            start_date = pd.Timestamp("2017-12-01")
            dt = start_date + pd.to_timedelta(df["TransactionDT"], unit='s')
            df["hour"] = dt.dt.hour
            df["dayofweek"] = dt.dt.dayofweek
            df["is_weekend"] = (df["dayofweek"] >= 5).astype(int)
            df["month"] = dt.dt.month
        if "TransactionAmt" in df.columns:
            df["log_TransactionAmt"] = np.log1p(df["TransactionAmt"])
        if "card1" in df.columns and "ieee_card1_stats" in agg_stats:
            df = df.merge(agg_stats["ieee_card1_stats"], on="card1", how="left")
        if "addr1" in df.columns and "ieee_addr1_stats" in agg_stats:
            df = df.merge(agg_stats["ieee_addr1_stats"], on="addr1", how="left")
        for col in ["P_emaildomain", "R_emaildomain"]:
            key = f"ieee_{col}_stats"
            if col in df.columns and key in agg_stats:
                df = df.merge(agg_stats[key], on=col, how="left")
    elif dataset_name == "EuropeanCard":
        if "Time" in df.columns:
            df["hour"] = (df["Time"] // 3600) % 24
            df["day"]  = df["Time"] // (24 * 3600)
            df["week"] = df["Time"] // (7 * 24 * 3600)
        if "Amount" in df.columns:
            df["log_Amount"] = np.log1p(df["Amount"])
    return df


def apply_preprocessing(raw_df, base_state, combo_state, dataset_name,
                        target_col="Fraud", training_columns=None):
    """
    Transform a raw validation/test DataFrame into the exact feature space
    of the corresponding resampled training CSV.
    """
    # ---- Separate target ----
    if target_col in raw_df.columns:
        y = raw_df[target_col].values
        X = raw_df.drop(columns=[target_col])
    else:
        y = None
        X = raw_df.copy()

    imp_cat = base_state["imputer_cat"]
    imp_num = base_state["imputer_num"]

    # ---- 1. Keep only columns the imputers were fitted on ----
    # This implicitly drops columns that had >5% missing in training.
    keep_cols = []
    if hasattr(imp_cat, 'feature_names_in_'):
        keep_cols.extend(imp_cat.feature_names_in_)
    if hasattr(imp_num, 'feature_names_in_'):
        keep_cols.extend(imp_num.feature_names_in_)
    X = X[[c for c in keep_cols if c in X.columns]].copy()

    # ---- 2. Impute ----
    cat_cols_imp = [c for c in imp_cat.feature_names_in_ if c in X.columns] if hasattr(imp_cat, 'feature_names_in_') else []
    num_cols_imp = [c for c in imp_num.feature_names_in_ if c in X.columns] if hasattr(imp_num, 'feature_names_in_') else []
    if cat_cols_imp:
        X[cat_cols_imp] = imp_cat.transform(X[cat_cols_imp])
    if num_cols_imp:
        X[num_cols_imp] = imp_num.transform(X[num_cols_imp])

    # ---- 3. Feature engineering (using saved agg_stats) ----
    X = _add_engineered_features(X, dataset_name, base_state["agg_stats"])
    # Replace NaN values that may appear from left‑joins on unseen keys
    X = X.fillna(0)

    # Drop high‑cardinality ID columns (same as training)
    for col in _ID_DROP_COLS:
        if col in X.columns:
            X.drop(columns=col, inplace=True)

    # ---- 4. Frequency encoding (saved freq_maps) ----
    freq_maps = base_state["freq_maps"]
    for col, fmap in freq_maps.items():
        if col in X.columns:
            X[col + "_freq"] = X[col].map(fmap).fillna(0)
            X.drop(columns=col, inplace=True)

    # ---- 5. Align exactly to the columns the encoder/scaler expect ----
    cat_cols = base_state["cat_cols"]
    num_cols = base_state["num_cols"]
    all_final_cols = cat_cols + num_cols
    X = X.reindex(columns=all_final_cols, fill_value=0)

    # ---- 6. Encode & scale ----
    if cat_cols:
        X[cat_cols] = base_state["encoder"].transform(X[cat_cols])
    if num_cols:
        X[num_cols] = base_state["scaler"].transform(X[num_cols])

    # ---- 7. Feature selection ----
    selector = combo_state.get("selector", None)
    if selector is not None:
        # Ensure column order matches what the selector was fitted on
        if hasattr(selector, 'feature_names_in_'):
            X = X.reindex(columns=list(selector.feature_names_in_), fill_value=0)
        mask = selector.get_support()
        X = X.loc[:, mask]

    # ---- 8. Final alignment to training column order ----
    if training_columns is not None:
        X = X.reindex(columns=training_columns, fill_value=0)

    return X

In [8]:
# =====================================================================
#  Data Loading & Column Alignment (updated – uses preprocessing states)
# =====================================================================

TARGET = CONFIG["TARGET_COL"]

def _variant_tag(train_path: Path, prefix: str) -> str:
    stem = train_path.stem
    tag  = stem.replace(prefix + "_", "").replace("train", "").strip("_-")
    return tag or "default"


def _coerce_numeric(df: pd.DataFrame) -> pd.DataFrame:
    non_numeric = df.select_dtypes(exclude=[np.number]).columns.tolist()
    if non_numeric:
        keep = [c for c in non_numeric if c == TARGET]
        drop = [c for c in non_numeric if c != TARGET]
        if drop:
            print(f"[load] dropping non-numeric cols: {drop[:5]}"
                  + (" ..." if len(drop) > 5 else ""))
            df = df.drop(columns=drop)
    return df


def load_variant(train_path: Path, val_path: Path, test_path: Path,
                 dataset_name: str, data_root: str = CONFIG["DATA_ROOT"],
                 target: str = TARGET):
    """
    Load train/val/test for one training variant.
    Val & test are raw CSVs (as saved by the preprocessing notebook).
    This function applies the exact same preprocessing pipeline (via the
    saved base_state & combo_state) to bring val & test into the training
    feature space.

    Returns
    -------
    X_train, y_train, X_val, y_val, X_test, y_test : pd.DataFrame / np.ndarray
    variant_tag : str
    """
    # ---- Parse variant info from training filename ------------------------
    info = parse_filename(train_path.stem)
    if info is None:
        raise ValueError(f"Cannot parse training filename: {train_path.name}")
    sampler, featsel, timestamp = info["sampler"], info["featsel"], info["timestamp"]

    # ---- Load raw CSVs ----------------------------------------------------
    df_tr  = pd.read_csv(train_path)
    df_val = pd.read_csv(val_path)
    df_te  = pd.read_csv(test_path)

    # Numeric coercion (defensive)
    df_tr  = _coerce_numeric(df_tr)

    if target not in df_tr.columns:
        raise ValueError(f"Target '{target}' missing in {train_path}")
    if target not in df_val.columns or target not in df_te.columns:
        raise ValueError(f"Target '{target}' missing in val/test for {train_path}")

    # ---- Load preprocessing states ----------------------------------------
    base_state_path = Path(data_root) / f"{dataset_name}_base_state.joblib"
    combo_state_path = Path(data_root) / f"{sampler}--{featsel}--{dataset_name}--{timestamp}_state.joblib"

    if not base_state_path.exists():
        raise FileNotFoundError(f"Base state not found: {base_state_path}")
    if not combo_state_path.exists():
        raise FileNotFoundError(f"Combo state not found: {combo_state_path}")

    base_state = joblib.load(base_state_path)
    combo_state = joblib.load(combo_state_path)

    # ---- Process raw validation & test ------------------------------------
    # Store training columns before dropping target (for alignment)
    train_cols = df_tr.drop(columns=[target]).columns.tolist()

    X_train = df_tr.drop(columns=[target]).reset_index(drop=True)
    y_train = df_tr[target].astype(int).values
    X_val = apply_preprocessing(
        df_val, base_state, combo_state, dataset_name,
        target_col=target, training_columns=train_cols
    )
    y_val = df_val[target].astype(int).values

    X_test = apply_preprocessing(
        df_te, base_state, combo_state, dataset_name,
        target_col=target, training_columns=train_cols
    )
    y_test = df_te[target].astype(int).values

    tag = _variant_tag(train_path, train_path.parent.name)
    return X_train, y_train, X_val, y_val, X_test, y_test, tag


# Smoke-test (optional, keep as before)
try:
    first_ds = next(iter(datasets))
    first_train = datasets[first_ds]["train"][0]
    print(f"[load] smoke test on {first_ds} / {first_train.name}")
    Xtr, ytr, Xv, yv, Xte, yte, tag = load_variant(
        first_train, datasets[first_ds]["val"], datasets[first_ds]["test"],
        dataset_name=first_ds
    )
    print(f"[load] X_train={Xtr.shape}  pos rate={ytr.mean():.4f}")
    print(f"[load] X_val  ={Xv.shape}   pos rate={yv.mean():.4f}")
    print(f"[load] X_test ={Xte.shape}  pos rate={yte.mean():.4f}")
    print(f"[load] variant tag = '{tag}'")
except Exception as e:
    print(f"[load][skip-smoke] {e}")

[load] smoke test on EuropeanCard / ADASYN--ANOVA_Percentile10--EuropeanCard--20260630_063233.csv
[load] X_train=(341010, 4)  pos rate=0.4999
[load] X_val  =(56961, 4)   pos rate=0.0010
[load] X_test =(56962, 4)  pos rate=0.0013
[load] variant tag = 'ADASYN--ANOVA_Percentile10--EuropeanCard--20260630_063233'


In [9]:
# =====================================================================
#  Deep Learning Model Builders (CNN / LSTM / RNN / GRU)
# =====================================================================

def _common_head(x, drop=0.3):
    x = layers.Dropout(drop)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(drop)(x)
    return layers.Dense(1, activation='sigmoid')(x)


def build_cnn(input_dim: int, random_state: int = GLOBAL_SEED) -> keras.Model:
    """One-dimensional CNN: extracts *local spatial patterns* across the feature vector."""
    inp = layers.Input(shape=(input_dim, 1), name="cnn_input")
    x = layers.Conv1D(32, kernel_size=3, padding='same', activation='relu')(inp)
    x = layers.Conv1D(64, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    out = _common_head(x, drop=0.3)
    model = models.Model(inp, out, name="CNN1D")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['AUC'])
    return model


def build_lstm(input_dim: int, random_state: int = GLOBAL_SEED) -> keras.Model:
    """LSTM: captures *recurrent sequential dependencies* across feature positions."""
    inp = layers.Input(shape=(input_dim, 1), name="lstm_input")
    x = layers.LSTM(32, return_sequences=False, dropout=0.2,
                    recurrent_dropout=0.0)(inp)
    out = _common_head(x, drop=0.3)
    model = models.Model(inp, out, name="LSTM")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['AUC'])
    return model


def build_vanilla_rnn(input_dim: int, random_state: int = GLOBAL_SEED) -> keras.Model:
    """Vanilla RNN (SimpleRNN): the lightweight recurrent alternative."""
    inp = layers.Input(shape=(input_dim, 1), name="rnn_input")
    x = layers.SimpleRNN(32, dropout=0.2)(inp)
    out = _common_head(x, drop=0.3)
    model = models.Model(inp, out, name="VanillaRNN")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['AUC'])
    return model


def build_gru(input_dim: int, random_state: int = GLOBAL_SEED) -> keras.Model:
    """GRU: lightweight recurrent model with gating; good speed/perf trade-off."""
    inp = layers.Input(shape=(input_dim, 1), name="gru_input")
    x = layers.GRU(32, dropout=0.2)(inp)
    out = _common_head(x, drop=0.3)
    model = models.Model(inp, out, name="GRU")
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss='binary_crossentropy', metrics=['AUC'])
    return model


DL_ARCH_BUILDERS: Dict[str, Callable[[int, int], keras.Model]] = {
    "cnn":   build_cnn,
    "lstm":  build_lstm,
    "rnn":   build_vanilla_rnn,
    "gru":   build_gru,
}
print(f"[dl] builders registered: {list(DL_ARCH_BUILDERS.keys())}")

[dl] builders registered: ['cnn', 'lstm', 'rnn', 'gru']


In [10]:
# =====================================================================
#  `KerasFraudClassifier` — sklearn-compatible wrapper
# =====================================================================
# Wraps any of the four DL builders into a sklearn-like estimator so it
# can be dropped into CV / pipelines / clones. Each `fit` call:
#   1) clears the previous TF graph,
#   2) reshapes 2D -> 3D (samples, features, 1),
#   3) carves a 10% stratified sub-validation for early-stopping,
#   4) trains with EarlyStopping(monitor='val_loss', patience, restore_best),
#   5) garbage-collects to avoid GPU/RAM bloat in CV loops.
# =====================================================================

@contextlib.contextmanager
def _tf_silence_stderr():
    """Redirect file descriptor 2 (stderr) to /dev/null for the block."""
    devnull_fd = os.open(os.devnull, os.O_WRONLY)
    saved_fd   = os.dup(2)
    os.dup2(devnull_fd, 2)
    os.close(devnull_fd)
    try:
        yield
    finally:
        os.dup2(saved_fd, 2)
        os.close(saved_fd)


class KerasFraudClassifier(BaseEstimator, ClassifierMixin):
    """Clean sklearn API wrapper around the 4 DL architectures above."""

    _ARCHS = ("cnn", "lstm", "rnn", "gru")

    def __init__(self,
                 architecture: str = "cnn",
                 epochs: int = 20,
                 batch_size: int = 256,
                 validation_split: float = 0.10,
                 patience: int = 4,
                 verbose: int = 0,
                 random_state: int = GLOBAL_SEED):
        self.architecture     = architecture
        self.epochs           = epochs
        self.batch_size       = batch_size
        self.validation_split = validation_split
        self.patience         = patience
        self.verbose          = verbose
        self.random_state     = random_state

    # ---- sklearn plumbing ---------------------------------------------
    def get_params(self, deep: bool = True) -> Dict[str, Any]:
        return {
            "architecture":     self.architecture,
            "epochs":           self.epochs,
            "batch_size":       self.batch_size,
            "validation_split": self.validation_split,
            "patience":         self.patience,
            "verbose":          self.verbose,
            "random_state":     self.random_state,
        }

    def set_params(self, **params) -> "KerasFraudClassifier":
        for k, v in params.items():
            setattr(self, k, v)
        return self

    # ---- core API ------------------------------------------------------
    def _reshape(self, X: np.ndarray) -> np.ndarray:
        """Convert 2D tabular input -> 3D tensor for Conv1D / RNN layers."""
        if X.ndim == 2:
            return X.reshape(X.shape[0], X.shape[1], 1).astype(np.float32)
        if X.ndim == 3:
            return X.astype(np.float32)
        raise ValueError(f"Expected 2D or 3D input, got shape {X.shape}")

    def fit(self, X, y):
        if self.architecture not in self._ARCHS:
            raise ValueError(f"Unknown architecture '{self.architecture}'")
        # reset graph so re-fits in CV don't accumulate
        K.clear_session()
        set_global_seed(self.random_state)

        Xr = self._reshape(np.asarray(X))
        yr = np.asarray(y).astype(int)

        # internal stratified sub-validation for early stopping
        if self.validation_split and self.validation_split > 0:
            Xt, Xv, yt, yv = train_test_split(
                Xr, yr, test_size=self.validation_split,
                stratify=yr, random_state=self.random_state
            )
        else:
            Xt, Xv, yt, yv = Xr, None, yr, None

        builder = DL_ARCH_BUILDERS[self.architecture]
        self.model_ = builder(input_dim=Xr.shape[1],
                              random_state=self.random_state)

        cb_list = [callbacks.EarlyStopping(
            monitor='val_loss', patience=self.patience,
            restore_best_weights=True, verbose=0
        )]

        fit_kwargs = dict(
            x=Xt, y=yt,
            epochs=self.epochs, batch_size=self.batch_size,
            callbacks=cb_list, verbose=self.verbose,
        )
        if Xv is not None:
            fit_kwargs["validation_data"] = (Xv, yv)
        else:
            fit_kwargs["validation_split"] = 0.0

        # ---- silence the benign `use_unbounded_threadpool` spam ------
        with _tf_silence_stderr():
            self.model_.fit(**fit_kwargs)

        # bookkeeping for sklearn
        self.classes_ = np.array([0, 1])
        self.n_features_in_ = Xr.shape[1]
        gc.collect()
        return self

    def predict_proba(self, X) -> np.ndarray:
        Xr = self._reshape(np.asarray(X))
        prob = self.model_.predict(Xr, verbose=0).flatten()
        prob = np.clip(prob, 1e-7, 1 - 1e-7)
        return np.vstack([1.0 - prob, prob]).T

    def predict(self, X) -> np.ndarray:
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


# quick smoke test (skip in CI by guarding)
try:
    _X_dummy = np.random.RandomState(0).randn(200, 12).astype(np.float32)
    _y_dummy = (np.random.RandomState(0).rand(200) > 0.85).astype(int)
    _clf = KerasFraudClassifier(architecture="cnn", epochs=2, verbose=0)
    _clf.fit(_X_dummy, _y_dummy)
    print(f"[dl wrapper] smoke test OK  "
          f"proba shape={_clf.predict_proba(_X_dummy).shape}  "
          f"classes={_clf.classes_.tolist()}")
    del _X_dummy, _y_dummy, _clf
    gc.collect()
except Exception as e:
    print(f"[dl wrapper][warn] smoke test skipped: {e}")

[dl wrapper] smoke test OK  proba shape=(200, 2)  classes=[0, 1]


In [11]:
# =====================================================================
#  Base Model Pool Factory (Level 0) – with unsupervised models added
# =====================================================================
def get_base_model_pool(n_features: int,
                        random_state: int = GLOBAL_SEED,
                        include_dl: bool = True) -> Dict[str, BaseEstimator]:
    """Returns the full Level-0 pool, including unsupervised anomaly detectors."""
    pool: Dict[str, BaseEstimator] = {}

    # ---- 1. Deep Learning ---------------------------------------------
    if include_dl:
        for arch in KerasFraudClassifier._ARCHS:
            pool[f"dl_{arch}"] = KerasFraudClassifier(
                architecture=arch,
                epochs=CONFIG["DL_EPOCHS"],
                batch_size=CONFIG["DL_BATCH_SIZE"],
                validation_split=CONFIG["DL_VAL_SPLIT"],
                patience=CONFIG["DL_PATIENCE"],
                verbose=0,
                random_state=random_state,
            )

    # ---- 2. Ultra-Fast -------------------------------------------------
    pool["lgb"] = LGBMClassifier(
        n_estimators=300, learning_rate=0.05,
        num_leaves=31, max_depth=-1,
        subsample=0.9, colsample_bytree=0.9,
        reg_alpha=0.0, reg_lambda=0.0,
        objective='binary', n_jobs=-1,
        random_state=random_state, verbose=-1,
    )
    pool["gaussian_nb"] = GaussianNB()
    pool["complement_nb"] = ImbPipeline([
        ("scale", MinMaxScaler()),
        ("clf",   ComplementNB()),
    ])
    pool["sgd_log"] = ImbPipeline([
        ("scale", StandardScaler()),
        ("clf",   SGDClassifier(loss="log_loss", penalty="elasticnet",
                                l1_ratio=0.15, alpha=1e-4,
                                max_iter=50, tol=1e-3,
                                random_state=random_state, n_jobs=-1)),
    ])
    pool["sgd_huber"] = ImbPipeline([
        ("scale", StandardScaler()),
        ("clf",   SGDClassifier(loss="modified_huber", penalty="l2",
                                alpha=1e-4, max_iter=50, tol=1e-3,
                                random_state=random_state, n_jobs=-1)),
    ])

    # ---- 3. Tree Boosting ----------------------------------------------
    pool["xgb"] = XGBClassifier(
        n_estimators=300, learning_rate=0.05,
        max_depth=6, subsample=0.9, colsample_bytree=0.9,
        tree_method="hist", eval_metric="auc",
        n_jobs=-1, random_state=random_state,
        verbosity=0, use_label_encoder=False,
    )
    if HAS_CATBOOST:
        pool["cat"] = CatBoostClassifier(
            iterations=300, learning_rate=0.05, depth=6,
            l2_leaf_reg=3.0, random_seed=random_state,
            verbose=0, allow_writing_files=False, thread_count=-1,
        )

    # ---- 4. Linear / Distance -----------------------------------------
    pool["logreg_en"] = ImbPipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(
            l1_ratio=0.5,          # elasticnet
            solver="saga",
            C=0.5,
            class_weight="balanced",
            max_iter=500,
            random_state=random_state
        )),
    ])
    pool["linsvc"] = ImbPipeline([
        ("scale", StandardScaler()),
        ("clf",   LinearSVC(C=0.5, class_weight="balanced",
                            max_iter=2000, random_state=random_state)),
    ])
    '''pool["knn"] = ImbPipeline([
        ("scale", StandardScaler()),
        ("clf",   KNeighborsClassifier(n_neighbors=min(15, max(3, n_features // 2)),
                                       n_jobs=-1, weights="distance")),
    ])''' # too much time

    # ---- 5. Unsupervised Anomaly Detectors (NEW) ------------------------
    # Isolation Forest – robust, fast, works out-of-the-box
    pool["isolation_forest"] = UnsupervisedAnomalyClassifier(
        IsolationForest(n_estimators=200, contamination=0.1,
                        random_state=random_state, n_jobs=-1)
    )
    '''# One-Class SVM – good for medium‑sized data
    pool["one_class_svm"] = UnsupervisedAnomalyClassifier(
        OneClassSVM(kernel='rbf', nu=0.1, gamma='scale')
    )
    # Local Outlier Factor (novelty mode) – density‑based
    pool["local_outlier_factor"] = UnsupervisedAnomalyClassifier(
        LocalOutlierFactor(novelty=True, contamination=0.1, n_jobs=-1)
    )'''#they take too much time

    return pool


# quick pool preview
_pool_preview = get_base_model_pool(n_features=20, include_dl=True)
print(f"[pool] {len(_pool_preview)} base estimators registered:")
for i, (name, est) in enumerate(_pool_preview.items(), 1):
    print(f"   {i:2d}. {name:22s} -> {type(est).__name__}")
del _pool_preview

[pool] 14 base estimators registered:
    1. dl_cnn                 -> KerasFraudClassifier
    2. dl_lstm                -> KerasFraudClassifier
    3. dl_rnn                 -> KerasFraudClassifier
    4. dl_gru                 -> KerasFraudClassifier
    5. lgb                    -> LGBMClassifier
    6. gaussian_nb            -> GaussianNB
    7. complement_nb          -> Pipeline
    8. sgd_log                -> Pipeline
    9. sgd_huber              -> Pipeline
   10. xgb                    -> XGBClassifier
   11. cat                    -> CatBoostClassifier
   12. logreg_en              -> Pipeline
   13. linsvc                 -> Pipeline
   14. isolation_forest       -> UnsupervisedAnomalyClassifier


In [12]:
# =====================================================================
#  Out-of-Fold (OOF) Feature Generator – PARALLELISED VERSION
# =====================================================================
# Uses joblib to train multiple (non‑DL) base models simultaneously.
# =====================================================================
from joblib import Parallel, delayed

def generate_oof_features(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_val:   pd.DataFrame,
    X_test:  pd.DataFrame,
    base_models: Dict[str, BaseEstimator],
    n_splits: int = 5,
    random_state: int = GLOBAL_SEED,
    verbose: int = 1,
    n_jobs: int = -1,          # <-- controls parallelism across models
):
    """
    Returns
    -------
    oof_df      : pd.DataFrame (n_train, n_models)   -- probabilities for the positive class
    val_df      : pd.DataFrame (n_val,   n_models)   -- K-fold averaged probabilities
    test_df     : pd.DataFrame (n_test,  n_models)   -- K-fold averaged probabilities
    """
    model_names = list(base_models.keys())
    n_train     = X_train.shape[0]
    n_val       = X_val.shape[0]
    n_test      = X_test.shape[0]
    n_models    = len(model_names)

    oof_proba   = np.zeros((n_train, n_models), dtype=np.float32)
    val_acc     = np.zeros((n_val,  n_models), dtype=np.float32)
    test_acc    = np.zeros((n_test, n_models), dtype=np.float32)
    failures: List[Tuple[int, str, str]] = []

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True,
                          random_state=random_state)

    for fold_idx, (tr_idx, ho_idx) in enumerate(skf.split(X_train, y_train), start=1):
        if verbose:
            print(f"   [fold {fold_idx}/{n_splits}]  "
                  f"train={len(tr_idx)}  holdout={len(ho_idx)}")
        Xtr = X_train.iloc[tr_idx]
        ytr = y_train[tr_idx]
        Xho = X_train.iloc[ho_idx]

        # ---- Split models into DL (run sequentially) and non‑DL (parallel) ----
        dl_names   = [n for n in model_names if isinstance(base_models[n], KerasFraudClassifier)]
        other_names = [n for n in model_names if n not in dl_names]

        # ---- Helper function for parallel workers ----
        def _fit_and_predict(name, est_clone):
            import warnings
            warnings.filterwarnings("ignore", message="X does not have valid feature names")
            try:
                est_clone.fit(Xtr, ytr)
                if hasattr(est_clone, "predict_proba"):
                    p_ho   = est_clone.predict_proba(Xho)[:, 1]
                    p_val  = est_clone.predict_proba(X_val)[:, 1]
                    p_test = est_clone.predict_proba(X_test)[:, 1]
                else:
                    # e.g. LinearSVC -> decision_function -> sigmoid
                    from scipy.special import expit
                    p_ho   = expit(est_clone.decision_function(Xho))
                    p_val  = expit(est_clone.decision_function(X_val.values))
                    p_test = expit(est_clone.decision_function(X_test.values))
                return name, p_ho, p_val, p_test, None
            except Exception as e:
                return name, None, None, None, str(e)

        # ---- 1. Train non‑DL models in parallel ----
        if other_names:
            # Build fresh clones OUTSIDE the parallel region (thread safety)
            clones = [(n, clone(base_models[n])) for n in other_names]
            results = Parallel(n_jobs=n_jobs, backend="loky")(
                delayed(_fit_and_predict)(n, cl) for n, cl in clones
            )
        else:
            results = []

        # ---- 2. Train DL models sequentially ----
        for name in dl_names:
            t0 = time.time()
            est = clone(base_models[name])
            try:
                est.fit(Xtr.values, ytr)
                p_ho   = est.predict_proba(Xho.values)[:, 1]
                p_val  = est.predict_proba(X_val.values)[:, 1]
                p_test = est.predict_proba(X_test.values)[:, 1]
                results.append((name, p_ho, p_val, p_test, None))
            except Exception as e:
                results.append((name, None, None, None, str(e)))
            finally:
                K.clear_session()
                gc.collect()
                if verbose >= 2:
                    print(f"        - {name:14s} {time.time()-t0:5.1f}s (DL)")

        # ---- 3. Collect results for this fold ----
        for name, p_ho, p_val, p_test, err in results:
            col = model_names.index(name)
            if err is not None:
                failures.append((fold_idx, name, err))
                oof_proba[ho_idx, col] = 0.5
                val_acc[:, col]   += 0.5
                test_acc[:, col]  += 0.5
                if verbose:
                    print(f"      ! model '{name}' failed on fold {fold_idx}: {err}")
            else:
                oof_proba[ho_idx, col] = p_ho
                val_acc[:, col]   += p_val
                test_acc[:, col]  += p_test

    # average K fold predictions for val & test
    val_df  = pd.DataFrame(val_acc  / n_splits, columns=model_names)
    test_df = pd.DataFrame(test_acc / n_splits, columns=model_names)
    oof_df  = pd.DataFrame(oof_proba, columns=model_names)

    if failures:
        print(f"\n[oof][warn] {len(failures)} model/fold failures (filled with 0.5 neutral):")
        for f, n, msg in failures:
            print(f"     fold {f} | {n:14s} | {msg[:80]}")

    return oof_df, val_df, test_df


print("[oof] parallelised generate_oof_features() ready.")

[oof] parallelised generate_oof_features() ready.


In [13]:
# =====================================================================
#  Level-1 Meta-Learner
# =====================================================================
# The Meta-Learner is trained on the FULL OOF matrix (n_train, n_models)
# against the true training labels. It learns optimal blending weights
# — including any negative coefficients that improve calibration.
# =====================================================================
from sklearn.neural_network import MLPClassifier   # add this import at the top

def train_meta_learner(oof_df, y_train, random_state=GLOBAL_SEED, kind="logreg_l2"):
    if kind == "logreg_l2":
        meta = LogisticRegression(l1_ratio=0, solver="saga", C=1.0, max_iter=1000, random_state=random_state)
    elif kind == "ridge":
        meta = RidgeClassifier(alpha=1.0, random_state=random_state)
    elif kind == "rf_shallow":
        meta = RandomForestClassifier(n_estimators=200, max_depth=3, min_samples_leaf=20,
                                      n_jobs=-1, random_state=random_state)
    elif kind == "gaussian_nb":
        meta = GaussianNB()
    elif kind == "mlp":
        meta = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
                             alpha=0.0001, max_iter=500, early_stopping=True,
                             validation_fraction=0.1, n_iter_no_change=10,
                             random_state=random_state)
    else:
        raise ValueError(f"Unknown meta-learner kind '{kind}'")
    meta.fit(oof_df.values, y_train)
    return meta


def meta_predict_proba(meta, X: pd.DataFrame) -> np.ndarray:
    """Uniform probability accessor for LogReg / RF. Ridge has no proba."""
    if hasattr(meta, "predict_proba"):
        return meta.predict_proba(X.values)[:, 1]
    # RidgeClassifier -> use decision_function + sigmoid
    from scipy.special import expit
    return expit(meta.decision_function(X.values))


print("[meta] train_meta_learner() + meta_predict_proba() ready.")

[meta] train_meta_learner() + meta_predict_proba() ready.


In [14]:
# =====================================================================
#  Evaluation Utility
# =====================================================================
# Computes Precision, Recall, F1, AUC-ROC and PR-AUC for both base models
# and the final Level-1 stacker. The Fraud class is the positive label.
# =====================================================================
def evaluate_predictions(name: str,
                         y_true: np.ndarray,
                         y_proba: np.ndarray,
                         threshold: float = 0.5,
                         verbose: bool = True) -> Dict[str, float]:
    """
    Compute the standard fraud-detection metrics on the positive (Fraud=1) class.

    Returns a dict of metrics for downstream aggregation.
    """
    y_pred = (y_proba >= threshold).astype(int)

    metrics = {
        "model":  name,
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall":    recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1":        f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "auc_roc":   roc_auc_score(y_true, y_proba),
        "auc_pr":    average_precision_score(y_true, y_proba),
        "n_pos":     int(y_true.sum()),
        "n_total":   int(len(y_true)),
    }

    if verbose:
        print(f"   [eval] {name:24s}  "
              f"P={metrics['precision']:.4f}  "
              f"R={metrics['recall']:.4f}  "
              f"F1={metrics['f1']:.4f}  "
              f"AUC-ROC={metrics['auc_roc']:.4f}  "
              f"PR-AUC={metrics['auc_pr']:.4f}")
    return metrics


def print_full_report(name: str, y_true: np.ndarray, y_proba: np.ndarray,
                      threshold: float = 0.5) -> None:
    """Pretty-print classification report + confusion matrix."""
    y_pred = (y_proba >= threshold).astype(int)
    print(f"\n--- {name} ---")
    print(classification_report(y_true, y_pred, digits=4, zero_division=0))
    print("Confusion matrix [[TN FP],[FN TP]]:")
    print(confusion_matrix(y_true, y_pred))


print("[eval] evaluate_predictions() + print_full_report() ready.")

[eval] evaluate_predictions() + print_full_report() ready.


In [15]:
# =====================================================================
#  Per-Variant Stacking Pipeline (UPDATED)
# =====================================================================
# Runs the entire L0 -> L1 chain on ONE (dataset, training-variant) tuple.
# Returns a metrics dict ready for aggregation, plus on-disk artifacts.
# =====================================================================
def run_variant_pipeline(
    dataset_name: str,
    train_path:   Path,
    val_path:     Path,
    test_path:    Path,
    config:       Dict[str, Any] = CONFIG,
    random_state: int = GLOBAL_SEED,
    meta_kind:    str = "logreg_l2",
    include_dl:   bool = True,
    save_artifacts: bool = True,
) -> Dict[str, Any]:
    """End-to-end pipeline for one training variant."""
    print(f"\n{'='*78}\n[dataset] {dataset_name}   "
          f"[variant] {train_path.name}\n{'='*78}")

    # ---- load (UPDATED call) --------------------------------------------
    Xtr_df, ytr, Xv_df, yv, Xte_df, yte, tag = load_variant(
        train_path, val_path, test_path,
        dataset_name=dataset_name,          # <-- NEW
        data_root=config["DATA_ROOT"],      # <-- NEW
        target=config["TARGET_COL"]
    )
    print(f"[load] X_train={Xtr_df.shape}  pos_rate={ytr.mean():.4f}  "
          f"X_val={Xv_df.shape}  X_test={Xte_df.shape}  tag='{tag}'")
    Xtr_df = safe_columns(Xtr_df)
    Xv_df  = safe_columns(Xv_df)
    Xte_df = safe_columns(Xte_df)

    # ---- L0 pool -------------------------------------------------------
    base_pool = get_base_model_pool(
        n_features=Xtr_df.shape[1],
        random_state=random_state,
        include_dl=include_dl,
    )
    print(f"[pool] {len(base_pool)} base estimators registered.")

    # ---- OOF generation ------------------------------------------------
    oof_df, val_meta_df, test_meta_df = generate_oof_features(
        X_train=Xtr_df, y_train=ytr,
        X_val=Xv_df,    X_test=Xte_df,
        base_models=base_pool,
        n_splits=config["N_SPLITS"],
        random_state=random_state,
        verbose=config["VERBOSE"],
    )

    # ---- base-model evaluation (sanity-check on VAL/TEST) ------------
    base_val_metrics, base_test_metrics = [], []
    print(f"\n[base] evaluating L0 models on val & test:")
    for col in oof_df.columns:
        try:
            base_val_metrics.append(
                evaluate_predictions(f"{col} (val)",  yv,  val_meta_df[col].values)
            )
            base_test_metrics.append(
                evaluate_predictions(f"{col} (test)", yte, test_meta_df[col].values)
            )
        except Exception as e:
            print(f"   [skip] {col}: {e}")

    # ---- Level-1 meta-learner -----------------------------------------
    print(f"\n[meta] training Level-1 ({meta_kind}) on OOF matrix "
          f"shape={oof_df.shape}")
    meta = train_meta_learner(oof_df, ytr, random_state=random_state, kind=meta_kind)

    meta_proba_val  = meta_predict_proba(meta, val_meta_df)
    meta_proba_test = meta_predict_proba(meta, test_meta_df)

    meta_val  = evaluate_predictions(f"STACK_{meta_kind} (val)",  yv,  meta_proba_val)
    meta_test = evaluate_predictions(f"STACK_{meta_kind} (test)", yte, meta_proba_test)

    # ---- coefficient / feature importance snapshot --------------------
    coef_map: Dict[str, float] = {}
    if hasattr(meta, "coef_"):
        coefs = np.ravel(meta.coef_)
        coef_map = {c: float(v) for c, v in zip(oof_df.columns, coefs)}
        sorted_coef = sorted(coef_map.items(), key=lambda kv: -abs(kv[1]))
        print("[meta] L1 weights (top-10 by |coef|):")
        for k, v in sorted_coef[:10]:
            print(f"     {k:14s} {v:+.4f}")
    elif hasattr(meta, "feature_importances_"):
        coef_map = {c: float(v) for c, v in zip(oof_df.columns, meta.feature_importances_)}

    # ---- artifact persistence (cache for downstream analysis) ---------
    if save_artifacts:
        out_dir = Path(config["OUTPUT_DIR"]) / dataset_name / tag
        out_dir.mkdir(parents=True, exist_ok=True)
        oof_df.to_csv(out_dir / "oof_features.csv", index=False)
        val_meta_df.to_csv(out_dir / "val_meta_features.csv", index=False)
        test_meta_df.to_csv(out_dir / "test_meta_features.csv", index=False)
        # full test predictions (proba + label) for downstream consumers
        pred_df = pd.DataFrame({
            "y_true": yte,
            "stack_proba": meta_proba_test,
            "stack_pred":  (meta_proba_test >= 0.5).astype(int),
        })
        pred_df.to_csv(out_dir / "test_predictions.csv", index=False)
        # save coefficients
        if coef_map:
            pd.DataFrame({"model": list(coef_map.keys()),
                          "weight": list(coef_map.values())}
                         ).sort_values("weight", key=lambda s: s.abs(),
                                       ascending=False
                         ).to_csv(out_dir / "meta_coefficients.csv", index=False)
        print(f"[save] artifacts -> {out_dir}")

    return {
        "dataset":           dataset_name,
        "variant":           tag,
        "pos_rate_train":    float(ytr.mean()),
        "n_base_models":     len(base_pool),
        "base_metrics_val":  base_val_metrics,
        "base_metrics_test": base_test_metrics,
        "meta_metrics_val":  meta_val,
        "meta_metrics_test": meta_test,
        "meta_coefficients": coef_map,
    }


print("[pipeline] run_variant_pipeline() ready.")

[pipeline] run_variant_pipeline() ready.


In [16]:
# =====================================================================
#  Full Multi-Dataset Orchestration
# =====================================================================
# Iterate over every (dataset x training-variant) combination, run the
# full pipeline, and aggregate metrics into one summary table.
# =====================================================================
def run_full_pipeline(
    datasets:    Dict[str, Dict[str, Any]] = datasets,
    config:      Dict[str, Any] = CONFIG,
    meta_kind:   str = "logreg_l2",
    include_dl:  bool = True,
    save_artifacts: bool = True,
    skip_existing: bool = True,          # <-- NEW
) -> pd.DataFrame:
    all_records: List[Dict[str, Any]] = []
    t_global = time.time()
    skipped_count = 0

    for ds_name, ds_info in datasets.items():
        for train_path in ds_info["train"]:
            # ---- skip if output already exists ---------------------------
            # Reproduce the tag that `run_variant_pipeline` would use
            tag = _variant_tag(train_path, train_path.parent.name)
            out_dir = Path(config["OUTPUT_DIR"]) / ds_name / tag
            if skip_existing and (out_dir / "test_predictions.csv").exists():
                if config.get("VERBOSE", 1):
                    print(f"[skip] {ds_name}/{tag}  — already processed")
                skipped_count += 1
                continue
            # -----------------------------------------------------------------

            res = run_variant_pipeline(
                dataset_name=ds_name,
                train_path=train_path,
                val_path=ds_info["val"],
                test_path=ds_info["test"],
                config=config,
                random_state=config["RANDOM_STATE"],
                meta_kind=meta_kind,
                include_dl=include_dl,
                save_artifacts=save_artifacts,
            )

            # ... rest of the flattening code (unchanged) ...

    results_df = pd.DataFrame(all_records)
    # Save the master summary (append or overwrite – current logic overwrites)
    out_path = Path(config["OUTPUT_DIR"]) / "results_summary.csv"
    results_df.to_csv(out_path, index=False)

    elapsed = (time.time() - t_global) / 60
    print(f"\n[done] full pipeline finished in {elapsed:.1f} min")
    if skip_existing:
        print(f"[done] processed {len(datasets)*len(ds_info['train'])} total variants, "
              f"skipped {skipped_count} already completed")
    print(f"[done] summary saved -> {out_path}")
    return results_df


# ---- Execute --------------------------------------------------------------
print("[run] launching full stacking pipeline across all datasets / variants ...")
results_df = run_full_pipeline()
print("\n[results] head of results_summary.csv:")
print(results_df.head(10).to_string(index=False))

[run] launching full stacking pipeline across all datasets / variants ...
[skip] EuropeanCard/ADASYN--ANOVA_Percentile10--EuropeanCard--20260630_063233  — already processed
[skip] EuropeanCard/ADASYN--ANOVA_Percentile20--EuropeanCard--20260630_063249  — already processed
[skip] EuropeanCard/ADASYN--ANOVA_Percentile30--EuropeanCard--20260630_063318  — already processed
[skip] EuropeanCard/ADASYN--ANOVA_Percentile50--EuropeanCard--20260630_063410  — already processed
[skip] EuropeanCard/ADASYN--ANOVA_k10--EuropeanCard--20260630_060816  — already processed
[skip] EuropeanCard/ADASYN--ANOVA_k15--EuropeanCard--20260630_061001  — already processed
[skip] EuropeanCard/ADASYN--ANOVA_k20--EuropeanCard--20260630_061358  — already processed
[skip] EuropeanCard/ADASYN--ANOVA_k30--EuropeanCard--20260630_061921  — already processed
[skip] EuropeanCard/ADASYN--ANOVA_k5--EuropeanCard--20260630_060725  — already processed
[skip] EuropeanCard/ADASYN--ANOVA_kall--EuropeanCard--20260630_062554  — already 

In [19]:
# =====================================================================
#  RECONSTRUCT RESULTS FROM SAVED ARTIFACTS & COMPARE (F1 PRIMARY)
# =====================================================================
# Reads per‑variant test predictions and base‑model meta‑features from disk,
# rebuilds the results table, and sorts by the stacker's F1 score.

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, average_precision_score

# Helper to compute metrics from true labels and predicted probabilities
def compute_metrics_from_proba(y_true, y_proba, model_name):
    y_pred = (y_proba >= 0.5).astype(int)
    return {
        "model": model_name,
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
        "auc_roc":   roc_auc_score(y_true, y_proba),
        "auc_pr":    average_precision_score(y_true, y_proba),
        "n_pos":     int(np.sum(y_true)),
        "n_total":   int(len(y_true)),
    }

# ---------------------------------------------------------------------
# 1. Rebuild results_df from disk
# ---------------------------------------------------------------------
output_root = Path(CONFIG["OUTPUT_DIR"])
all_records = []

for test_pred_path in output_root.glob("*/*/test_predictions.csv"):
    variant_dir = test_pred_path.parent
    dataset = variant_dir.parent.name
    variant = variant_dir.name

    pred_df = pd.read_csv(test_pred_path)
    y_true = pred_df["y_true"].values
    stack_proba = pred_df["stack_proba"].values

    meta_path = variant_dir / "test_meta_features.csv"
    if not meta_path.exists():
        print(f"[reconstruct] missing {meta_path} for {dataset}/{variant}, skipping")
        continue
    meta_df = pd.read_csv(meta_path)

    # Base models (L0)
    for col in meta_df.columns:
        try:
            metrics = compute_metrics_from_proba(y_true, meta_df[col].values, col)
            all_records.append({
                "dataset": dataset,
                "variant": variant,
                "layer":   "L0",
                **metrics
            })
        except Exception as e:
            print(f"[reconstruct] error computing metrics for {col} in {dataset}/{variant}: {e}")

    # Stacker (L1)
    try:
        stack_metrics = compute_metrics_from_proba(y_true, stack_proba, "STACK")
        all_records.append({
            "dataset": dataset,
            "variant": variant,
            "layer":   "L1",
            **stack_metrics
        })
    except Exception as e:
        print(f"[reconstruct] error computing stack metrics for {dataset}/{variant}: {e}")

results_df = pd.DataFrame(all_records)
print(f"[reconstruct] Rebuilt results_df with {len(results_df)} rows from {output_root}")

# ---------------------------------------------------------------------
# 2. Comparison function – sorted by stack F1 (primary metric)
# ---------------------------------------------------------------------
def stack_vs_best_base(df: pd.DataFrame) -> pd.DataFrame:
    """
    Return, per dataset/variant, F1 of the stacker vs the best L0.
    Sorted by the stacker's F1 score (descending).
    """
    required_cols = {"dataset", "variant", "layer", "model", "f1", "auc_roc"}
    if df is None or df.empty or not required_cols.issubset(df.columns):
        print("[compare] No valid results to compare (missing columns or empty DataFrame).")
        return pd.DataFrame(columns=[
            "dataset", "variant", "best_l0_model", "best_l0_f1", "stack_f1",
            "lift_f1", "best_l0_auc", "stack_auc", "lift_auc"
        ])

    rows = []
    for (ds, var), group in df.groupby(["dataset", "variant"]):
        l0 = group[group["layer"] == "L0"]
        l1_rows = group[group["layer"] == "L1"]
        if l0.empty or l1_rows.empty:
            continue

        l1 = l1_rows.iloc[0]
        best_l0 = l0.loc[l0["f1"].idxmax()]   # best L0 by F1

        rows.append({
            "dataset":        ds,
            "variant":        var,
            "best_l0_model":  best_l0["model"],
            "best_l0_f1":     best_l0["f1"],
            "stack_f1":       l1["f1"],
            "lift_f1":        l1["f1"] - best_l0["f1"],
            "best_l0_auc":    best_l0["auc_roc"],
            "stack_auc":      l1["auc_roc"],
            "lift_auc":       l1["auc_roc"] - best_l0["auc_roc"],
        })

    # Sort by stack F1 descending (primary), then lift_f1 descending
    return pd.DataFrame(rows).sort_values(
        by=["stack_f1", "lift_f1"], ascending=[False, False]
    )

# ---------------------------------------------------------------------
# 3. Run comparison and display table
# ---------------------------------------------------------------------
lift_df = stack_vs_best_base(results_df)
print("\n[compare] L1 stacker vs best L0 base model (TEST F1 PRIMARY):")
print(lift_df.to_string(index=False))

[reconstruct] Rebuilt results_df with 8400 rows from trained

[compare] L1 stacker vs best L0 base model (TEST F1 PRIMARY):
     dataset                                                                    variant    best_l0_model  best_l0_f1  stack_f1   lift_f1  best_l0_auc  stack_auc  lift_auc
EuropeanCard          EditedNearestNeighbours--ANOVA_k15--EuropeanCard--20260630_061047              cat    0.814286  0.844444  0.030159     0.983595   0.975659 -0.007936
EuropeanCard             EditedNearestNeighbours--MI_k10--EuropeanCard--20260630_060934              cat    0.844444  0.842105 -0.002339     0.979009   0.981583  0.002574
EuropeanCard             EditedNearestNeighbours--MI_k30--EuropeanCard--20260630_062326              cat    0.823529  0.835821  0.012291     0.984670   0.984857  0.000188
EuropeanCard EditedNearestNeighbours--ANOVA_Percentile20--EuropeanCard--20260630_063301              xgb    0.820896  0.833333  0.012438     0.958027   0.962239  0.004212
EuropeanCard         

In [22]:
# =====================================================================
#  Per-Dataset Leaderboards: Top-10 L0 Models & Top-3 L1 Stackers
# =====================================================================
# Uses results_df (already reconstructed) to display detailed rankings.

import pandas as pd
from pathlib import Path

# Helper to format metric columns nicely
def display_leaderboard(df: pd.DataFrame, title: str, top_n: int = 10):
    print("\n" + "=" * 100)
    print(f"{title}")
    print("=" * 100)
    if df.empty:
        print("No entries found.")
        return

    # Columns to show (all metrics)
    cols = [
        "dataset", "variant", "model",
        "precision", "recall", "f1", "auc_roc", "auc_pr",
        "n_pos", "n_total"
    ]
    # Keep only columns that exist
    display_cols = [c for c in cols if c in df.columns]
    print(df[display_cols].to_string(index=False))
    print("-" * 100)

# Ensure results_df exists and is non-empty
if "results_df" not in globals() or results_df.empty:
    print("[leaderboard] results_df is empty. Re-run the reconstruction cell first.")
else:
    # ---- 1. Prepare per-dataset outputs ---------------------------------
    datasets_list = sorted(results_df["dataset"].unique())
    output_dir = Path(CONFIG["OUTPUT_DIR"]) / "leaderboards"
    output_dir.mkdir(parents=True, exist_ok=True)

    for ds in datasets_list:
        ds_df = results_df[results_df["dataset"] == ds]

        # ---------- Top 10 L0 (single base models) ----------
        l0_df = ds_df[ds_df["layer"] == "L0"]
        top10_l0 = l0_df.sort_values(by="f1", ascending=False).head(10)

        # ---------- Top 3 L1 (stacker variants) ----------
        l1_df = ds_df[ds_df["layer"] == "L1"]
        top3_l1 = l1_df.sort_values(by="f1", ascending=False).head(3)

        # ---------- Display ----------
        display_leaderboard(top10_l0, f"Dataset: {ds}  –  TOP 10 SINGLE MODELS (L0) by F1", top_n=10)
        display_leaderboard(top3_l1, f"Dataset: {ds}  –  TOP 3 STACKING VARIANTS (L1) by F1", top_n=3)

        # ---------- Save to CSV ----------
        top10_l0.to_csv(output_dir / f"{ds}_top10_L0_by_f1.csv", index=False)
        top3_l1.to_csv(output_dir / f"{ds}_top3_L1_by_f1.csv", index=False)
        print(f"[save] {ds}: top10 L0 -> {output_dir / f'{ds}_top10_L0_by_f1.csv'}")
        print(f"[save] {ds}: top3 L1 -> {output_dir / f'{ds}_top3_L1_by_f1.csv'}")

    print("\n[leaderboard] All per-dataset rankings saved to:", output_dir)


Dataset: EuropeanCard  –  TOP 10 SINGLE MODELS (L0) by F1
     dataset                                                                   variant  model  precision   recall       f1  auc_roc   auc_pr  n_pos  n_total
EuropeanCard            EditedNearestNeighbours--MI_k10--EuropeanCard--20260630_060934    cat   0.950000 0.760000 0.844444 0.979009 0.795160     75    56962
EuropeanCard            EditedNearestNeighbours--MI_k20--EuropeanCard--20260630_061718    xgb   0.982143 0.733333 0.839695 0.981989 0.804826     75    56962
EuropeanCard                         TomekLinks--MI_k15--EuropeanCard--20260630_061224    xgb   0.964912 0.733333 0.833333 0.979757 0.802137     75    56962
EuropeanCard                         TomekLinks--MI_k10--EuropeanCard--20260630_060927    xgb   0.964912 0.733333 0.833333 0.963484 0.793990     75    56962
EuropeanCard            EditedNearestNeighbours--MI_k20--EuropeanCard--20260630_061718    cat   0.964912 0.733333 0.833333 0.980615 0.804575     75    56962

**Key observation:**  
Stacking (hybrid ensemble) does **not** automatically outperform the best single model.  
Its success depends on **carefully selecting diverse base models** trained with **different preprocessing strategies**.  

In our results:
- IEEE-CIS: stacking improved F1 over the best L0 (0.4648 vs 0.4525).
- EuropeanCard: stacking matched but did not exceed the best L0 (0.8444 vs 0.8444).
- Sparkov: stacking underperformed the best L0 (0.7794 vs 0.8012).

Therefore, hybrid models should be designed deliberately, not treated as a guaranteed improvement.

In [24]:
# =====================================================================
#  Reproducibility & Cleanup
# =====================================================================
# Final housekeeping. Always run this cell at the end of the session.
# =====================================================================
K.clear_session()
gc.collect()

# Snapshot config used for this run
with open(Path(CONFIG["OUTPUT_DIR"]) / "run_config.json", "w") as f:
    json.dump({k: str(v) if isinstance(v, Path) else v
               for k, v in CONFIG.items()}, f, indent=2)

print(f"[cleanup] TF graph cleared, GC ran.")
print(f"[cleanup] run_config.json saved -> {CONFIG['OUTPUT_DIR']}")
print("[done] Hybrid Stacking pipeline complete.")

[cleanup] TF graph cleared, GC ran.
[cleanup] run_config.json saved -> ./trained
[done] Hybrid Stacking pipeline complete.


## Notebook Summary

This notebook delivered a **production-grade, leakage-free Stacked Generalization** pipeline
for fraud detection across three datasets and multiple training variants.

### What was built

| Stage | Component | Highlights |
|-------|-----------|------------|
| **Data** | Auto-discovery loader | Aligns val/test columns to each variant |
| **L0 — Deep** | CNN, LSTM, RNN, GRU | Sklearn-compatible wrapper, early stopping on 10% sub-val, per-fold session reset |
| **L0 — Ultra-Fast** | LightGBM, GaussianNB, ComplementNB, SGD (log/huber) | Histogram binning, elastic-net penalty |
| **L0 — Boosting** | XGBoost, CatBoost | Hist tree method, balanced class weights |
| **L0 — Linear** | LogReg(L1/ElasticNet), LinearSVC, KNN | Wrapped in `ImbPipeline` + `StandardScaler` so scaling is fit per-fold |
| **L1 Meta** | LogReg(L2) / Ridge / Shallow RF | Trained on full OOF matrix; weights logged |
| **Eval** | Precision / Recall / F1 / AUC-ROC / PR-AUC | Computed for every L0 and the L1 stacker |
| **Artifacts** | `oof_features.csv`, `val_meta_features.csv`, `test_meta_features.csv`, `test_predictions.csv`, `meta_coefficients.csv`, `results_summary.csv` | Drop-in for downstream analysis |

### Anti-leakage audit checklist
- [x] All preprocessing wrapped in `Imblearn.pipeline.Pipeline` — fit per fold
- [x] `StratifiedKFold` on training data only
- [x] Validation set **never used** for any selection / decision
- [x] Test set touched **exactly once** for final evaluation
- [x] DL models cleared session + GC between folds
- [x] Global seeding across `random`, `numpy`, `tensorflow`

### Where to plug this in
The `OOF matrices` and `test_predictions.csv` files in
`/workspace/stacking_artifacts/<dataset>/<variant>/` are formatted for direct
consumption by the downstream LLM-explanation wrapper (`explainer.ipynb`) or any
further error / SHAP analysis script.